# Temporal IM-Loss SNN Colab Runner

This notebook runs the repository command-line workflow on a Colab GPU runtime. It keeps training and evaluation logic in the Python source files and uses the notebook only for environment setup, launching experiments, and collecting summaries.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys


def run(args, env=None):
    args = [str(arg) for arg in args]
    print("$", " ".join(args))
    subprocess.run(args, check=True, env=env)


## Repository Setup

If this notebook is already inside the cloned repository, leave `REPO_URL` empty. If starting from a fresh Colab notebook, set `REPO_URL` to the Git URL first.

In [ ]:
REPO_URL = ""
REPO_DIR = Path("/content/IM-Loss-Temporal-SNN")

if REPO_URL:
    if not REPO_DIR.exists():
        run(["git", "clone", REPO_URL, REPO_DIR])
    os.chdir(REPO_DIR)
    run(["git", "pull", "--ff-only"])
else:
    if not Path("train_temporal.py").exists():
        raise RuntimeError("Set REPO_URL or open this notebook from the repository root.")
    REPO_DIR = Path.cwd()

print("repo:", Path.cwd())


In [ ]:
run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])
run([sys.executable, "train_temporal.py", "--help"])


## Persistent Run Directory

Colab local storage is temporary. Mount Drive and write checkpoints, metrics, and summaries there.

In [ ]:
RUNS_ROOT = "/content/drive/MyDrive/im_snn_runs"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(RUNS_ROOT).mkdir(parents=True, exist_ok=True)
except Exception as exc:
    print("Drive mount unavailable; using local runs directory:", exc)
    RUNS_ROOT = "runs"

print("runs root:", RUNS_ROOT)


## Shared Experiment Settings

In [ ]:
os.environ.update({
    "RUNS_ROOT": RUNS_ROOT,
    "DATASET": "Randman",
    "EPOCHS": "100",
    "BATCH_SIZE": "64",
    "NUM_WORKERS": "2",
    "HIDDEN_SIZE": "256",
    "NUM_LAYERS": "1",
    "READOUT": "max_membrane",
})

if os.environ.get("COLAB_GPU"):
    os.environ.pop("NO_CUDA", None)
else:
    os.environ["NO_CUDA"] = "1"


## Quick Checks

In [ ]:
run([sys.executable, "scripts/smoke_temporal.py"])
run([
    sys.executable,
    "train_temporal.py",
    "--dataset", "Randman",
    "--epochs", "1",
    "--max_train_batches", "1",
    "--max_eval_batches", "1",
])


## Three-Condition Randman Comparison

In [ ]:
FULL_THREE_WAY = False
three_way_tasks = range(15) if FULL_THREE_WAY else [0, 5, 10]

for task_id in three_way_tasks:
    run([sys.executable, "scripts/run_temporal_task.py", "ff_output_3way", task_id])


## Lambda Sweep

In [ ]:
RUN_LAMBDA_SWEEP = False
os.environ["IM_LAMBDAS"] = "0.0,0.0001,0.0003,0.001,0.003,0.01"
os.environ["SEEDS"] = "2020,42,123"
os.environ["LAMBDA_SWEEP_IM_LOSS_TYPE"] = "threshold"

if RUN_LAMBDA_SWEEP:
    lambda_count = len([x for x in os.environ["IM_LAMBDAS"].split(",") if x.strip()])
    seed_count = len([x for x in os.environ["SEEDS"].split(",") if x.strip()])
    for task_id in range(lambda_count * seed_count):
        run([sys.executable, "scripts/run_temporal_task.py", "ff_output_lambda_sweep", task_id])


## Summaries

In [ ]:
summary_dir = Path(RUNS_ROOT) / "summary"
summary_dir.mkdir(parents=True, exist_ok=True)

summary_cmd = [
    sys.executable,
    "summarize_ff_output_3way.py",
    "--dataset", "Randman",
    "--run_root", RUNS_ROOT,
    "--output_dir", summary_dir,
]
if not FULL_THREE_WAY:
    summary_cmd.append("--allow_partial")
run(summary_cmd)

if RUN_LAMBDA_SWEEP:
    run([
        sys.executable,
        "summarize_lambda_sweep.py",
        "--dataset", "Randman",
        "--loss_type", os.environ["LAMBDA_SWEEP_IM_LOSS_TYPE"],
        "--run_root", RUNS_ROOT,
        "--output_dir", summary_dir,
    ])


## SHD Run Template

Place `shd_train.h5` and `shd_test.h5` in the Drive path below, then run the cells after switching `DATASET` to `SHD`.

In [ ]:
RUN_SHD_TEMPLATE = False

if RUN_SHD_TEMPLATE:
    os.environ.update({
        "DATASET": "SHD",
        "DATA_ROOT": "/content/drive/MyDrive/datasets/SHD",
        "EPOCHS": "100",
        "BATCH_SIZE": "64",
    })
    run([sys.executable, "scripts/run_temporal_task.py", "ff_output_3way", 0, "--dry-run"])
